In [27]:
import pandas as pd
import numpy as np
import sys
import os
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sys.path.append('../../../utils')
from LabelEncode import auto_label_encode
from SubmissionHelper import save_submission

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [28]:
def feature_engineering(df):
    df_copy = df.copy()
    df_copy['Evap_Index'] = (df_copy['Temperature_C'] * df_copy['Wind_Speed_kmh']) / (df_copy['Humidity'] + 1e-5)
    df_copy['Water_Stress'] = df_copy['Rainfall_mm'] / (df_copy['Soil_Moisture'] + 1e-5)
    df_copy['Crop_Soil'] = df_copy['Crop_Type'].astype(str) + "_" + df_copy['Soil_Type'].astype(str)
    return df_copy

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

In [31]:
print(X_all.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 23 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Soil_Type                630000 non-null  object 
 1   Soil_pH                  630000 non-null  float64
 2   Soil_Moisture            630000 non-null  float64
 3   Organic_Carbon           630000 non-null  float64
 4   Electrical_Conductivity  630000 non-null  float64
 5   Temperature_C            630000 non-null  float64
 6   Humidity                 630000 non-null  float64
 7   Rainfall_mm              630000 non-null  float64
 8   Sunlight_Hours           630000 non-null  float64
 9   Wind_Speed_kmh           630000 non-null  float64
 10  Crop_Type                630000 non-null  object 
 11  Crop_Growth_Stage        630000 non-null  object 
 12  Season                   630000 non-null  object 
 13  Irrigation_Type          630000 non-null  object 
 14  Wate

In [7]:
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
inv_target_map = {0: 'Low', 1: 'Medium', 2: 'High'}
y = train_fe['Irrigation_Need'].map(target_map)

# XGBoost 需要做标签编码
train_xgb, test_xgb, _ = auto_label_encode(
    train_fe, test_fe, target_col='Irrigation_Need', exclude_cols=['id']
)
X_xgb = train_xgb.drop(['id', 'Irrigation_Need'], axis=1)
X_test_xgb = test_xgb.drop(['id'], axis=1)

# CatBoost 需要object编码
X_cat = train_fe.drop(['id', 'Irrigation_Need'], axis=1).fillna('None')
X_test_cat = test_fe.drop(['id'], axis=1).fillna('None')
cat_features = list(X_cat.select_dtypes(include=['object']).columns)

✅ Categorical columns encoded: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 'Crop_Soil']


In [9]:
# 划分 XGB 数据
X_tr_xgb, X_val_xgb, y_tr, y_val = train_test_split(X_xgb, y, test_size=0.2, random_state=42, stratify=y)
# 划分 Cat 核心数据
X_tr_cat, X_val_cat = X_cat.iloc[X_tr_xgb.index], X_cat.iloc[X_val_xgb.index]

# 训练 XGBoost
xgb_model = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, tree_method='hist')
xgb_model.fit(X_tr_xgb, y_tr, eval_set=[(X_val_xgb, y_val)], verbose=False)

# 训练 CatBoost
cat_model = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, cat_features=cat_features, task_type='GPU', verbose=False)
cat_model.fit(X_tr_cat, y_tr, eval_set=(X_val_cat, y_val))

In [34]:
# 给出概率预测
val_xgb_probs = xgb_model.predict_proba(X_val_xgb)
val_cat_probs = cat_model.predict_proba(X_val_cat)
# 提交答案
val_xgb_preds = np.argmax(val_xgb_probs, axis=1)
val_cat_preds = np.argmax(val_cat_probs, axis=1)
# 和正确答案比对 改卷
score_xgb = accuracy_score(y_val, val_xgb_preds)
score_cat = accuracy_score(y_val, val_cat_preds)

print(f"XGB Val Accuracy: {score_xgb:.5f}")
print(f"Cat Val Accuracy: {score_cat:.5f}")

XGB Val Accuracy: 0.98459
Cat Val Accuracy: 0.98299


In [22]:
# 获取概率预测 (predict_proba)
xgb_probs = xgb_model.predict_proba(X_test_xgb)
cat_probs = cat_model.predict_proba(X_test_cat)

# 设定权重
final_probs = (0.99 * xgb_probs) + (0.01 * cat_probs)
final_preds = np.argmax(final_probs, axis=1)

final_labels = [inv_target_map[p] for p in final_preds]

In [23]:
save_submission(final_labels, test, 'id', 'Irrigation_Need', prefix='ensemble_xgb_cat_9901')

------------------------------
✅ [SUCCESS] Submission file generated!
📍 Location: D:\Kaggle-Learning\02-Tabular-Data\playground-series-s6e4\submissions\ensemble_xgb_cat_9901_0404_1859.csv
📊 Shape: (270000, 2)
------------------------------


'../submissions\\ensemble_xgb_cat_9901_0404_1859.csv'